In [1]:
import numpy as np

In [1]:
import carla
import time
client = carla.Client('localhost', 2000)
world = client.get_world()

def find_obj(world, keyword, offset=0):
    env_objs = world.get_environment_objects()
    objs = []

    for env_obj in env_objs:
        if keyword in env_obj.name:
            _env_obj = env_obj
            _env_obj.transform.location.z += offset
            objs.append(_env_obj)

    return objs

spawn_point = find_obj(world, "truck_spawn", 0.5)[0]

truck_bp = world.get_blueprint_library().find("vehicle.daf.dafxf")
trailer_bp = world.get_blueprint_library().find("vehicle.trailer.trailer")

In [22]:
truck_parking = find_obj(world, "truck_parking1")[0].transform
cone_nums = [28, 24, 70, 55]
cones = []

for num in cone_nums:
    cones.append(find_obj(world, f"SM_ConstructionCone{num}")[0].transform)

In [23]:
parking_loc = np.array([truck_parking.location.x, truck_parking.location.y])
for cone in cones:
    cone_loc = np.array([cone.location.x, cone.location.y])
    distance = np.linalg.norm(cone_loc - parking_loc)

    print(distance)

34.93365089569254
34.194529733134004
30.597705889641055
31.441717706805555


In [5]:
from parking_env4 import *
env = ParkingLotEnv()

In [ ]:
env.reset()
world.get_actor(world.get_actors().filter("vehicle.daf*")[0].id).apply_control(carla.VehicleControl(brake=0.5, reverse=False))

Destroying 0 actors...
Truck Spawned.
Trailer Spawned and Attached.
Spawned 24 actors (vehicles + sensors).


: 

In [81]:
env.destroy_actors()

Destroying 24 actors...


In [4]:
obj = find_obj(world, "truck_parking1")[0]
center = obj.transform.location
radius = 2.5
num_points = 72

# Draw a 1 meter radius circle around the object's center.
for i in range(num_points):
    theta1 = 2.0 * np.pi * i / num_points
    theta2 = 2.0 * np.pi * (i + 1) / num_points
    p1 = carla.Location(
        x=center.x + radius * np.cos(theta1),
        y=center.y + radius * np.sin(theta1),
        z=center.z + 0.1
    )
    p2 = carla.Location(
        x=center.x + radius * np.cos(theta2),
        y=center.y + radius * np.sin(theta2),
        z=center.z + 0.1
    )
    world.debug.draw_line(
        p1,
        p2,
        thickness=0.08,
        color=carla.Color(r=0, g=255, b=255),
        life_time=10.0
    )

obj

In [ ]:
for sensor in world.get_actors().filter("sensor.other.obstacle*"):
    sensor_transform = sensor.get_transform()

    world.debug.draw_arrow(
        sensor_transform.location,
        sensor_transform.location + sensor_transform.get_forward_vector() * 1.3,
        thickness=0.1,
        arrow_size=0.1,
        color=carla.Color(r=255, g=125, b=0),
        life_time=10
    )

In [ ]:
import numpy as np

# Get the trailer parking object
parking_point = find_obj(world, 'trailer_parking')[0]
parking_location = parking_point.transform.location

# Get the trailer actor
trailer = world.get_actors().filter("vehicle.trailer*")[0]
trailer_transform = trailer.get_transform()

# Calculate trailer's back end position
trailer_yaw_rad = np.deg2rad(trailer_transform.rotation.yaw)
trailer_backward = np.array([-np.cos(trailer_yaw_rad), -np.sin(trailer_yaw_rad)])
trailer_bb = trailer.bounding_box
trailer_back_end = np.array([trailer_transform.location.x - 5.0, trailer_transform.location.y]) + trailer_backward * trailer_bb.extent.x

# Draw line from trailer back end to parking spot
world.debug.draw_line(
    carla.Location(x=trailer_back_end[0], y=trailer_back_end[1], z=0.5),
    carla.Location(x=parking_location.x, y=parking_location.y, z=0.5),
    thickness=0.1,
    color=carla.Color(r=0, g=255, b=0),
    life_time=50.0
)